In [1]:
import torch
import joblib
import numpy as np

In [2]:
class GaussianMF(torch.nn.Module):
    def __init__(self, c, sigma):
        super().__init__()
        self.c = torch.nn.Parameter(torch.tensor(float(c)))
        self.sigma = torch.nn.Parameter(torch.tensor(float(sigma)))
    def forward(self, x):
        return torch.exp(-0.5 * ((x - self.c) / self.sigma)**2)

class ANFIS(torch.nn.Module):
    def __init__(self, n_inputs, mfs_per_input):
        super().__init__()
        self.n_inputs = n_inputs
        self.m = mfs_per_input
        self.mf_layer = torch.nn.ModuleList([
            torch.nn.ModuleList([GaussianMF(0.5, 0.2) for _ in range(mfs_per_input)])
            for _ in range(n_inputs)
        ])
        self.n_rules = mfs_per_input ** n_inputs
        self.linear = torch.nn.Linear(n_inputs, self.n_rules, bias=True)
    def forward(self, x):
        batch_size = x.size(0)
        memberships = torch.stack([
            torch.stack([mf(x[:, i]) for mf in self.mf_layer[i]], dim=-1)
            for i in range(self.n_inputs)
        ], dim=1)
        rule_strengths = memberships[:,0,:]
        for i in range(1, self.n_inputs):
            rule_strengths = rule_strengths.unsqueeze(2) * memberships[:,i,:].unsqueeze(1)
            rule_strengths = rule_strengths.reshape(batch_size, -1)
        w_normalized = rule_strengths / (rule_strengths.sum(dim=1, keepdim=True) + 1e-6)
        linear_output = self.linear(x)
        output = (w_normalized * linear_output).sum(dim=1, keepdim=True)
        return output

# 1. Inicijalizuj model
model_loaded = ANFIS(n_inputs=4, mfs_per_input=3)

# 2. Učitaj parametre
model_loaded.load_state_dict(torch.load("anfis_model.pt"))
model_loaded.eval()

# 3. Učitaj skalere
scaler_X = joblib.load("scaler_X.save")
scaler_y = joblib.load("scaler_y.save")

In [3]:
def predict_anfis(model, X_input, scaler_X, scaler_y):
    """
    X_input : np.array shape [n_samples, 4] (Coop, Adap, Forg, Stoch)
    vraća: np.array shape [n_samples, 1] sa predikcijom Output
    """
    Xn = scaler_X.transform(X_input)
    X_tensor = torch.tensor(Xn, dtype=torch.float32)

    with torch.no_grad():
        y_pred = model(X_tensor).numpy()

    y_pred_rescaled = scaler_y.inverse_transform(y_pred)
    return y_pred_rescaled

In [4]:
X_new = np.array([
    [0.7, 0.3, 0.5, 0.2],
    [0.1, 0.9, 0.4, 0.6]
])

y_pred = predict_anfis(model_loaded, X_new, scaler_X, scaler_y)
print("Predikcija ANFIS:", y_pred)

Predikcija ANFIS: [[0.9443366 ]
 [0.94570374]]
